## modified codet5

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, T5EncoderModel, T5Config
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')


# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
print("Loading data...")
train_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/train_original.csv")
test_data = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/test_original.csv")
tag_vocab = pd.read_csv("/home/aman_swaraj/Downloads/Codelite/unique_tags.csv")["Tag"].tolist()

# Encode labels
label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)
train_data["EncodedTags"] = label_encoder.transform(train_data["language"])
test_data["EncodedTags"] = label_encoder.transform(test_data["language"])
num_classes = len(tag_vocab)
print(f"Number of classes: {num_classes}")
print(f"Classes: {tag_vocab}")

In [ ]:
print("Initializing CodeT5 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")
print(f"Tokenizer class: {type(tokenizer).__name__}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

def tokenize_code(code, max_length=512):
    """
    Tokenize code for CodeT5 encoder.
    CodeT5 uses RobertaTokenizer which is a byte-level BPE tokenizer.
    """
    return tokenizer(
        code,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

In [ ]:
sample_code = "def hello_world():\n    print('Hello, World!')"
sample_tokens = tokenize_code(sample_code)
print(f"\nSample tokenization - input_ids shape: {sample_tokens['input_ids'].shape}")
print(f"Sample tokenization - attention_mask shape: {sample_tokens['attention_mask'].shape}")


In [ ]:
class CodeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = tokenize_code(row["code"])
        
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedTags"], dtype=torch.long)
        }

train_dataset = CodeDataset(train_data)
test_dataset = CodeDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"\nTrain batches: {len(train_loader)}, Test batches: {len(test_loader)}")


In [ ]:
class CodeT5Stage1(nn.Module):
    """
    Stage 1: Basic CodeT5 encoder with mean pooling and classifier.
    """
    def __init__(self, num_classes):
        super().__init__()
        self.codet5 = T5EncoderModel.from_pretrained("Salesforce/codet5-base")
        hidden_size = self.codet5.config.hidden_size
        
        self.classifier = nn.Linear(hidden_size, num_classes)
        print(f"\nStage 1 Model - Hidden size: {hidden_size}")
        
    def forward(self, input_ids, attention_mask):
        outputs = self.codet5(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        token_embeddings = outputs.last_hidden_state
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_embeddings = torch.sum(token_embeddings * attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
        pooled_output = sum_embeddings / sum_mask
        
        logits = self.classifier(pooled_output)
        return logits


In [ ]:
class CodeT5_RI_Transformer(nn.Module):
    """
    Stage 2: CodeT5 with layer representation information extraction.
    """
    def __init__(self, num_classes):
        super().__init__()
        
        self.codet5 = T5EncoderModel.from_pretrained(
            "Salesforce/codet5-base",
            output_hidden_states=True
        )
        
        hidden_size = self.codet5.config.hidden_size
        num_layers = self.codet5.config.num_layers
        
        print(f"\nStage 2 Model Config:")
        print(f"  - Hidden size: {hidden_size}")
        print(f"  - Number of layers: {num_layers}")
        print(f"  - Dtype: {self.codet5.dtype}")
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=8,
            dim_feedforward=hidden_size * 4,
            dropout=0.1,
            batch_first=True,
            activation='gelu'
        )
        
        self.layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        
        self.attention_fc = nn.Linear(hidden_size, hidden_size)
        self.context_vector = nn.Parameter(torch.randn(hidden_size))
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_classes)
        )
        
        self._init_weights()
        
    def _init_weights(self):
        """Initialize weights for newly added layers"""
        for module in [self.layer_transformer, self.attention_fc, self.classifier]:
            if hasattr(module, 'weight') and module.weight is not None:
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight)
                    if module.bias is not None:
                        nn.init.zeros_(module.bias)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.codet5(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        hidden_states = outputs.hidden_states
        
        encoder_hidden_states = hidden_states[1:]
        
        batch_size = input_ids.size(0)
        sequence_length = input_ids.size(1)
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).float()
        layerwise_embeddings = []
        
        for layer_idx, layer_hidden in enumerate(encoder_hidden_states):
            sum_embeddings = torch.sum(layer_hidden * attention_mask_expanded, dim=1)
            sum_mask = torch.clamp(attention_mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_embeddings / sum_mask
            layerwise_embeddings.append(pooled)
        
        layer_sequence = torch.stack(layerwise_embeddings, dim=1)
        
        transformed_layers = self.layer_transformer(layer_sequence)
        
        u = torch.tanh(self.attention_fc(transformed_layers))
        
        scores = torch.matmul(u, self.context_vector)
        
        alpha = torch.softmax(scores, dim=1)
        
        weighted_output = torch.sum(transformed_layers * alpha.unsqueeze(-1), dim=1)
        
        logits = self.classifier(weighted_output)
        
        return logits

In [ ]:
#  TRAINING STAGE 1 
print("\n" + "="*60)
print("STAGE 1 TRAINING: Basic CodeT5 Encoder")
print("="*60)

stage1_model = CodeT5Stage1(num_classes).to(device)

total_params = sum(p.numel() for p in stage1_model.parameters())
trainable_params = sum(p.numel() for p in stage1_model.parameters() if p.requires_grad)
print(f"Stage 1 - Total parameters: {total_params:,}")
print(f"Stage 1 - Trainable parameters: {trainable_params:,}")

optimizer_stage1 = optim.AdamW(stage1_model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

epochs_stage1 = 1
stage1_model.train()

for epoch in range(epochs_stage1):
    total_loss = 0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader, 1):
        optimizer_stage1.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = stage1_model(input_ids, attention_mask)
        
        loss = criterion(logits, labels)
        
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(stage1_model.parameters(), max_norm=1.0)
        optimizer_stage1.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            accuracy = correct / total
            print(f"  Stage1 - Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, Acc: {accuracy:.4f}")
    
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"[Stage 1] Epoch {epoch+1} Complete")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Training Accuracy: {epoch_accuracy:.4f}")

print("\nSaving Stage 1 CodeT5 encoder weights...")
torch.save(
    stage1_model.codet5.state_dict(),
    "/home/aman_swaraj/Downloads/Codelite/codet5_stage1_weights.pt"
)
print("Stage 1 weights saved successfully.")

In [ ]:
#  TRAINING STAGE 2 
print("\n" + "="*60)
print("STAGE 2 TRAINING: Layer Representation Transformer")
print("="*60)

stage2_model = CodeT5_RI_Transformer(num_classes).to(device)

print("Loading Stage 1 CodeT5 encoder weights into Stage 2 model...")
try:
    stage2_model.codet5.load_state_dict(
        torch.load("/home/aman_swaraj/Downloads/Codelite/codet5_stage1_weights.pt")
    )
    print("✓ Stage 1 weights loaded successfully")
except Exception as e:
    print(f"✗ Error loading weights: {e}")
    print("Initializing with fresh weights...")

for param in stage2_model.codet5.parameters():
    param.requires_grad = False

total_params_stage2 = sum(p.numel() for p in stage2_model.parameters())
trainable_params_stage2 = sum(p.numel() for p in stage2_model.parameters() if p.requires_grad)
frozen_params_stage2 = total_params_stage2 - trainable_params_stage2

print(f"\nStage 2 Parameter Summary:")
print(f"  - Total parameters: {total_params_stage2:,}")
print(f"  - Trainable parameters: {trainable_params_stage2:,}")
print(f"  - Frozen parameters: {frozen_params_stage2:,}")
print(f"  - Trainable percentage: {trainable_params_stage2/total_params_stage2*100:.2f}%")

optimizer_stage2 = optim.AdamW(
    filter(lambda p: p.requires_grad, stage2_model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

epochs_stage2 = 2
stage2_model.train()

for epoch in range(epochs_stage2):
    total_loss = 0
    correct = 0
    total = 0
    
    for step, batch in enumerate(train_loader, 1):
        optimizer_stage2.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits = stage2_model(input_ids, attention_mask)
        
        loss = criterion(logits, labels)
        
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, stage2_model.parameters()),
            max_norm=1.0
        )
        optimizer_stage2.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            accuracy = correct / total
            print(f"  Stage2 - Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, Acc: {accuracy:.4f}")
    
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total
    print(f"[Stage 2] Epoch {epoch+1} Complete")
    print(f"  Average Loss: {avg_loss:.4f}")
    print(f"  Training Accuracy: {epoch_accuracy:.4f}")

print("\nSaving complete Stage 2 model...")
torch.save(
    {
        'model_state_dict': stage2_model.state_dict(),
        'config': stage2_model.codet5.config.to_dict(),
        'num_classes': num_classes
    },
    "/home/aman_swaraj/Downloads/Codelite/codet5_full_RI_transformer.pt"
)
print("Stage 2 model saved successfully.")

In [ ]:
print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

stage2_model.eval()
all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        
        logits = stage2_model(input_ids, attention_mask)
        
        predictions = torch.argmax(logits, dim=1).cpu().numpy()
        
        all_predictions.extend(predictions)
        all_true_labels.extend(labels)
        
        if batch_idx % 10 == 0:
            print(f"  Processed {batch_idx}/{len(test_loader)} test batches")

accuracy = accuracy_score(all_true_labels, all_predictions)
print(f"\nTest Accuracy: {accuracy:.4f}")

print("\n" + "-"*60)
print("Classification Report:")
print("-"*60)
report = classification_report(
    all_true_labels,
    all_predictions,
    target_names=tag_vocab,
    digits=4,
    zero_division=0
)
print(report)

In [ ]:
print("\nSaving results to CSV...")
results_df = pd.DataFrame({
    "Code": test_data["code"],
    "True_Label": label_encoder.inverse_transform(all_true_labels),
    "Predicted_Label": label_encoder.inverse_transform(all_predictions),
    "Correct": [true == pred for true, pred in zip(all_true_labels, all_predictions)]
})

results_df.to_csv(
    "/home/aman_swaraj/Downloads/Codelite/codet5_RI_transformer_results.csv",
    index=False
)

print(f"Results saved to: /home/aman_swaraj/Downloads/Codelite/codet5_RI_transformer_results.csv")
print(f"Total test samples: {len(results_df)}")
print(f"Correct predictions: {results_df['Correct'].sum()}")
print(f"Test accuracy: {results_df['Correct'].sum() / len(results_df):.4f}")

In [ ]:
print("\n" + "="*60)
print("EVALUATING STAGE 1 MODEL ON TEST SET")
print("="*60)

stage1_model.eval()

stage1_predictions = []
stage1_true_labels = []

stage1_logits_list = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader, 1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        
        logits = stage1_model(input_ids, attention_mask)
        
        stage1_logits_list.append(logits.cpu())
        
        predictions = torch.argmax(logits, dim=1).cpu().numpy()
        
        stage1_predictions.extend(predictions)
        stage1_true_labels.extend(labels)
        
        if batch_idx % 10 == 0:
            print(f"  Stage1 - Processed {batch_idx}/{len(test_loader)} test batches")

stage1_accuracy = accuracy_score(stage1_true_labels, stage1_predictions)
print(f"\nStage 1 Model Test Accuracy: {stage1_accuracy:.4f}")

print("\n" + "-"*60)
print("Stage 1 Model Classification Report:")
print("-"*60)
stage1_report = classification_report(
    stage1_true_labels,
    stage1_predictions,
    target_names=tag_vocab,
    digits=4,
    zero_division=0
)
print(stage1_report)

print("\n" + "="*60)
print("COMPARISON: STAGE 1 vs STAGE 2 PERFORMANCE")
print("="*60)

stage2_accuracy = accuracy_score(all_true_labels, all_predictions)

print(f"\nAccuracy Comparison:")
print(f"  Stage 1 (Basic CodeT5): {stage1_accuracy:.4f}")
print(f"  Stage 2 (RI Transformer): {stage2_accuracy:.4f}")
print(f"  Difference: {stage2_accuracy - stage1_accuracy:+.4f}")

from collections import Counter

def get_confusion_matrix(true_labels, pred_labels, num_classes):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    for t, p in zip(true_labels, pred_labels):
        cm[t, p] += 1
    return cm

stage1_cm = get_confusion_matrix(stage1_true_labels, stage1_predictions, num_classes)
stage2_cm = get_confusion_matrix(all_true_labels, all_predictions, num_classes)

print(f"\nPer-Class Accuracy Comparison:")
print("-" * 50)
print(f"{'Class':<20} {'Stage1':<10} {'Stage2':<10} {'Diff':<10}")
print("-" * 50)

for i, class_name in enumerate(tag_vocab):
    stage1_correct = stage1_cm[i, i].item()
    stage1_total = stage1_cm[i, :].sum().item()
    stage1_acc = stage1_correct / stage1_total if stage1_total > 0 else 0
    
    stage2_correct = stage2_cm[i, i].item()
    stage2_total = stage2_cm[i, :].sum().item()
    stage2_acc = stage2_correct / stage2_total if stage2_total > 0 else 0
    
    diff = stage2_acc - stage1_acc
    
    print(f"{class_name:<20} {stage1_acc:.4f}    {stage2_acc:.4f}    {diff:+.4f}")

print("\nSaving Stage 1 evaluation results...")

stage1_results_df = pd.DataFrame({
    "Code": test_data["code"],
    "True_Label": label_encoder.inverse_transform(stage1_true_labels),
    "Predicted_Label_Stage1": label_encoder.inverse_transform(stage1_predictions),
    "Predicted_Label_Stage2": label_encoder.inverse_transform(all_predictions),
    "Stage1_Correct": [true == pred for true, pred in zip(stage1_true_labels, stage1_predictions)],
    "Stage2_Correct": [true == pred for true, pred in zip(all_true_labels, all_predictions)]
})

stage1_results_df["Both_Stages_Agree"] = stage1_results_df["Predicted_Label_Stage1"] == stage1_results_df["Predicted_Label_Stage2"]

stage1_results_df.to_csv(
    "/home/aman_swaraj/Downloads/Codelite/codet5_stage1_evaluation_results.csv",
    index=False
)

print(f"Stage 1 results saved to: /home/aman_swaraj/Downloads/Codelite/codet5_stage1_evaluation_results.csv")

print("\n" + "="*60)
print("ANALYSIS OF PREDICTION CHANGES BETWEEN STAGES")
print("="*60)

total_samples = len(stage1_results_df)
agreement_count = stage1_results_df["Both_Stages_Agree"].sum()
disagreement_count = total_samples - agreement_count

print(f"\nPrediction Agreement:")
print(f"  Total samples: {total_samples}")
print(f"  Both stages agree: {agreement_count} ({agreement_count/total_samples*100:.2f}%)")
print(f"  Stages disagree: {disagreement_count} ({disagreement_count/total_samples*100:.2f}%)")

disagreements_df = stage1_results_df[~stage1_results_df["Both_Stages_Agree"]]

if len(disagreements_df) > 0:
    print(f"\nWhen stages disagree:")
    
    stage2_improvements = disagreements_df[disagreements_df["Stage2_Correct"] & ~disagreements_df["Stage1_Correct"]]
    
    stage2_regressions = disagreements_df[~disagreements_df["Stage2_Correct"] & disagreements_df["Stage1_Correct"]]
    
    both_wrong_diff = disagreements_df[~disagreements_df["Stage2_Correct"] & ~disagreements_df["Stage1_Correct"]]
    
    print(f"  Stage 2 corrected Stage 1's mistakes: {len(stage2_improvements)}")
    print(f"  Stage 2 made new mistakes: {len(stage2_regressions)}")
    print(f"  Both wrong but different predictions: {len(both_wrong_diff)}")
    
    if len(disagreements_df) > 0:
        disagreements_df.to_csv(
            "/home/aman_swaraj/Downloads/Codelite/codet5_stage_disagreements.csv",
            index=False
        )
        print(f"\nDetailed disagreement analysis saved to: /home/aman_swaraj/Downloads/Codelite/codet5_stage_disagreements.csv")

print("\n" + "="*60)
print("CONFIDENCE ANALYSIS FOR BOTH STAGES")
print("="*60)

all_stage1_logits = torch.cat(stage1_logits_list, dim=0)

stage1_probs = torch.softmax(all_stage1_logits, dim=1)
stage1_confidences = torch.max(stage1_probs, dim=1)[0].numpy()

print("\nStage 1 Confidence Statistics:")
print(f"  Mean confidence: {stage1_confidences.mean():.4f}")
print(f"  Std confidence: {stage1_confidences.std():.4f}")
print(f"  Min confidence: {stage1_confidences.min():.4f}")
print(f"  Max confidence: {stage1_confidences.max():.4f}")

stage1_correct_mask = np.array([true == pred for true, pred in zip(stage1_true_labels, stage1_predictions)])

correct_confidences = stage1_confidences[stage1_correct_mask]
wrong_confidences = stage1_confidences[~stage1_correct_mask]

print(f"\nStage 1 Confidence by Prediction Correctness:")
print(f"  Correct predictions (n={len(correct_confidences)}):")
print(f"    Mean confidence: {correct_confidences.mean():.4f}")
print(f"    Std confidence: {correct_confidences.std():.4f}")
print(f"  Wrong predictions (n={len(wrong_confidences)}):")
print(f"    Mean confidence: {wrong_confidences.mean():.4f}")
print(f"    Std confidence: {wrong_confidences.std():.4f}")

print("\n" + "="*60)
print("SAVING COMPREHENSIVE EVALUATION REPORT")
print("="*60)

summary_report = {
    "Model": "CodeT5 Two-Stage Tuning with RI Extraction",
    "Test Set Size": len(test_data),
    "Number of Classes": num_classes,
    "Stage 1 Accuracy": float(stage1_accuracy),
    "Stage 2 Accuracy": float(stage2_accuracy),
    "Accuracy Improvement": float(stage2_accuracy - stage1_accuracy),
    "Prediction Agreement": float(agreement_count / total_samples),
    "Stage 1 Mean Confidence": float(stage1_confidences.mean()),
    "Stage 1 Correct Prediction Mean Confidence": float(correct_confidences.mean()),
    "Stage 1 Wrong Prediction Mean Confidence": float(wrong_confidences.mean())
}

summary_df = pd.DataFrame([summary_report])
summary_df.to_csv(
    "/home/aman_swaraj/Downloads/Codelite/codet5_two_stage_summary.csv",
    index=False
)

print(f"Summary report saved to: /home/aman_swaraj/Downloads/Codelite/codet5_two_stage_summary.csv")

print("\n" + "="*60)
print("STAGE 1 EVALUATION COMPLETE")
print("="*60)
print(f"✓ Stage 1 model evaluated on {len(test_data)} test samples")
print(f"✓ Stage 1 accuracy: {stage1_accuracy:.4f}")
print(f"✓ Stage 2 accuracy: {stage2_accuracy:.4f}")
print(f"✓ Comparison saved to multiple CSV files")
print("="*60)

In [ ]:
import numpy as np
print("\n" + "="*60)
print("CONFIDENCE ANALYSIS FOR BOTH STAGES")
print("="*60)

all_stage1_logits = torch.cat(stage1_logits_list, dim=0)

stage1_probs = torch.softmax(all_stage1_logits, dim=1)
stage1_confidences = torch.max(stage1_probs, dim=1)[0].numpy()

print("\nStage 1 Confidence Statistics:")
print(f"  Mean confidence: {stage1_confidences.mean():.4f}")
print(f"  Std confidence: {stage1_confidences.std():.4f}")
print(f"  Min confidence: {stage1_confidences.min():.4f}")
print(f"  Max confidence: {stage1_confidences.max():.4f}")

tage1_correct_mask = np.array([true == pred for true, pred in zip(stage1_true_labels, stage1_predictions)])

correct_confidences = stage1_confidences[stage1_correct_mask]
wrong_confidences = stage1_confidences[~stage1_correct_mask]

print(f"\nStage 1 Confidence by Prediction Correctness:")
print(f"  Correct predictions (n={len(correct_confidences)}):")
print(f"    Mean confidence: {correct_confidences.mean():.4f}")
print(f"    Std confidence: {correct_confidences.std():.4f}")
print(f"  Wrong predictions (n={len(wrong_confidences)}):")
print(f"    Mean confidence: {wrong_confidences.mean():.4f}")
print(f"    Std confidence: {wrong_confidences.std():.4f}")

print("\n" + "="*60)
print("SAVING COMPREHENSIVE EVALUATION REPORT")
print("="*60)

summary_report = {
    "Model": "CodeT5 Two-Stage Tuning with RI Extraction",
    "Test Set Size": len(test_data),
    "Number of Classes": num_classes,
    "Stage 1 Accuracy": float(stage1_accuracy),
    "Stage 2 Accuracy": float(stage2_accuracy),
    "Accuracy Improvement": float(stage2_accuracy - stage1_accuracy),
    "Prediction Agreement": float(agreement_count / total_samples),
    "Stage 1 Mean Confidence": float(stage1_confidences.mean()),
    "Stage 1 Correct Prediction Mean Confidence": float(correct_confidences.mean()),
    "Stage 1 Wrong Prediction Mean Confidence": float(wrong_confidences.mean())
}

summary_df = pd.DataFrame([summary_report])
summary_df.to_csv(
    "/home/aman_swaraj/Downloads/Codelite/codet5_two_stage_summary.csv",
    index=False
)

print(f"Summary report saved to: /home/aman_swaraj/Downloads/Codelite/codet5_two_stage_summary.csv")

print("\n" + "="*60)
print("STAGE 1 EVALUATION COMPLETE")
print("="*60)
print(f"✓ Stage 1 model evaluated on {len(test_data)} test samples")
print(f"✓ Stage 1 accuracy: {stage1_accuracy:.4f}")
print(f"✓ Stage 2 accuracy: {stage2_accuracy:.4f}")
print(f"✓ Comparison saved to multiple CSV files")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*60)

import numpy as np
from scipy import stats
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar

print("\n1. McNemar's Test (Paired Comparison)")
print("-" * 50)



stage1_correct = np.array([true == pred for true, pred in zip(stage1_true_labels, stage1_predictions)])
stage2_correct = np.array([true == pred for true, pred in zip(all_true_labels, all_predictions)])

contingency_table = np.zeros((2, 2), dtype=int)

for s1, s2 in zip(stage1_correct, stage2_correct):
    contingency_table[int(s1), int(s2)] += 1

print(f"Contingency Table:")
print(f"                Stage2 Correct  Stage2 Wrong")
print(f"Stage1 Correct    {contingency_table[1,1]:>7d}        {contingency_table[1,0]:>7d}")
print(f"Stage1 Wrong      {contingency_table[0,1]:>7d}        {contingency_table[0,0]:>7d}")

result = mcnemar(contingency_table, exact=False, correction=True)
print(f"\nMcNemar's Test Results:")
print(f"  χ² statistic: {result.statistic:.4f}")
print(f"  p-value: {result.pvalue:.6f}")

if result.pvalue < 0.05:
    print(f"  Result: SIGNIFICANT difference (p < 0.05)")
    if contingency_table[0,1] > contingency_table[1,0]:
        print(f"  Interpretation: Stage 2 significantly outperforms Stage 1")
    else:
        print(f"  Interpretation: Stage 1 significantly outperforms Stage 2")
else:
    print(f"  Result: NO significant difference (p ≥ 0.05)")
    print(f"  Interpretation: Both models perform similarly")

print("\n" + "-" * 50)
print("2. Paired T-Test on Per-Sample Correctness")
print("-" * 50)

stage1_correct_binary = stage1_correct.astype(int)
stage2_correct_binary = stage2_correct.astype(int)

t_stat, p_value = stats.ttest_rel(stage1_correct_binary, stage2_correct_binary)

print(f"Paired T-Test Results:")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.6f}")

mean_stage1 = stage1_correct_binary.mean()
mean_stage2 = stage2_correct_binary.mean()
mean_diff = mean_stage2 - mean_stage1
print(f"  Mean Stage 1 accuracy: {mean_stage1:.4f}")
print(f"  Mean Stage 2 accuracy: {mean_stage2:.4f}")
print(f"  Mean difference: {mean_diff:.4f}")

n = len(stage1_correct_binary)
se = np.std(stage1_correct_binary - stage2_correct_binary) / np.sqrt(n)
ci_lower = mean_diff - 1.96 * se
ci_upper = mean_diff + 1.96 * se
print(f"  95% CI for difference: [{ci_lower:.4f}, {ci_upper:.4f}]")

print("\n" + "-" * 50)
print("3. Cohen's Kappa (Agreement Between Models)")
print("-" * 50)

kappa = cohen_kappa_score(stage1_predictions, all_predictions)
print(f"Cohen's Kappa: {kappa:.4f}")

if kappa < 0:
    interpretation = "No agreement"
elif kappa <= 0.20:
    interpretation = "Slight agreement"
elif kappa <= 0.40:
    interpretation = "Fair agreement"
elif kappa <= 0.60:
    interpretation = "Moderate agreement"
elif kappa <= 0.80:
    interpretation = "Substantial agreement"
else:
    interpretation = "Almost perfect agreement"

print(f"Interpretation: {interpretation}")

print("\n" + "-" * 50)
print("4. Permutation Test (Non-parametric)")
print("-" * 50)

def permutation_test(x, y, n_permutations=10000):
    """Perform permutation test for paired data"""
    observed_diff = np.mean(y) - np.mean(x)
    diffs = []
    
    for _ in range(n_permutations):
        swap_mask = np.random.randint(0, 2, len(x)) * 2 - 1
        perm_x = x.copy()
        perm_y = y.copy()
        
        swap_idx = np.where(swap_mask == -1)[0]
        perm_x[swap_idx], perm_y[swap_idx] = perm_y[swap_idx], perm_x[swap_idx]
        
        diffs.append(np.mean(perm_y) - np.mean(perm_x))
    
    diffs = np.array(diffs)
    p_value = np.sum(np.abs(diffs) >= np.abs(observed_diff)) / n_permutations
    
    return observed_diff, p_value

obs_diff, perm_p_value = permutation_test(stage1_correct_binary, stage2_correct_binary, n_permutations=5000)
print(f"Permutation Test Results:")
print(f"  Observed difference: {obs_diff:.4f}")
print(f"  p-value: {perm_p_value:.6f}")

if perm_p_value < 0.05:
    print(f"  Result: Statistically significant (p < 0.05)")
else:
    print(f"  Result: Not statistically significant (p ≥ 0.05)")

print("\n" + "-" * 50)
print("5. Effect Size Measures")
print("-" * 50)


diff_scores = stage2_correct_binary - stage1_correct_binary
cohen_d = np.mean(diff_scores) / np.std(diff_scores, ddof=1)
print(f"Cohen's d (paired): {cohen_d:.4f}")

if abs(cohen_d) < 0.2:
    d_interpretation = "Very small effect"
elif abs(cohen_d) < 0.5:
    d_interpretation = "Small effect"
elif abs(cohen_d) < 0.8:
    d_interpretation = "Medium effect"
else:
    d_interpretation = "Large effect"
print(f"Interpretation: {d_interpretation}")


odds_ratio = (contingency_table[0,1] + 0.5) / (contingency_table[1,0] + 0.5)  # Add 0.5 for continuity correction
print(f"\nOdds Ratio (Stage2 correct when Stage1 wrong vs reverse): {odds_ratio:.4f}")
print(f"Interpretation: Stage 2 is {odds_ratio:.2f} times more likely to be correct when Stage 1 is wrong")

print("\n" + "-" * 50)
print("6. Bootstrap Confidence Intervals")
print("-" * 50)

def bootstrap_ci(data1, data2, n_bootstrap=10000, ci_level=0.95):
    """Calculate bootstrap CI for accuracy difference"""
    n = len(data1)
    diffs = []
    
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        boot1 = data1[idx]
        boot2 = data2[idx]
        diff = boot2.mean() - boot1.mean()
        diffs.append(diff)
    
    diffs = np.array(diffs)
    alpha = 1 - ci_level
    lower = np.percentile(diffs, 100 * alpha/2)
    upper = np.percentile(diffs, 100 * (1 - alpha/2))
    
    return lower, upper, diffs

lower_ci, upper_ci, bootstrap_diffs = bootstrap_ci(
    stage1_correct_binary, 
    stage2_correct_binary,
    n_bootstrap=5000
)

print(f"Bootstrap 95% CI for accuracy difference: [{lower_ci:.4f}, {upper_ci:.4f}]")
print(f"Bootstrap mean difference: {np.mean(bootstrap_diffs):.4f}")
print(f"Bootstrap standard error: {np.std(bootstrap_diffs):.4f}")

print("\n" + "-" * 50)
print("7. Per-Class Statistical Tests")
print("-" * 50)

print("\nPer-Class McNemar Tests (Significant differences only):")
print("-" * 70)
print(f"{'Class':<20} {'S1 Acc':<8} {'S2 Acc':<8} {'Diff':<8} {'p-value':<10} {'Significant'}")
print("-" * 70)

significant_classes = []
for class_idx, class_name in enumerate(tag_vocab):
    class_indices = [i for i, label in enumerate(stage1_true_labels) if label == class_idx]
    
    if len(class_indices) < 10:  
        continue
    
    s1_class_preds = [stage1_predictions[i] for i in class_indices]
    s2_class_preds = [all_predictions[i] for i in class_indices]
    true_class_labels = [stage1_true_labels[i] for i in class_indices]
    
    s1_correct = [1 if s1_class_preds[i] == true_class_labels[i] else 0 for i in range(len(class_indices))]
    s2_correct = [1 if s2_class_preds[i] == true_class_labels[i] else 0 for i in range(len(class_indices))]
    
    ct = np.zeros((2, 2), dtype=int)
    for s1, s2 in zip(s1_correct, s2_correct):
        ct[int(s1), int(s2)] += 1
    
    if ct[0,1] + ct[1,0] > 0:  
        result_class = mcnemar(ct, exact=False, correction=True)
        p_value_class = result_class.pvalue
        
        s1_acc = np.mean(s1_correct)
        s2_acc = np.mean(s2_correct)
        diff = s2_acc - s1_acc
        
        if p_value_class < 0.05:
            significant_classes.append((class_name, s1_acc, s2_acc, diff, p_value_class))
            star = "***"
        elif p_value_class < 0.10:
            star = "*"
        else:
            star = ""
        
        if p_value_class < 0.10:  
            print(f"{class_name:<20} {s1_acc:.4f}   {s2_acc:.4f}   {diff:+.4f}   {p_value_class:.6f}   {star}")

print("\n" + "-" * 50)
print("8. Statistical Power Analysis")
print("-" * 50)

from statsmodels.stats.power import TTestIndPower

effect_size = cohen_d
n = len(stage1_correct_binary)
alpha = 0.05

power_analysis = TTestIndPower()
power = power_analysis.power(effect_size=abs(effect_size), nobs1=n, alpha=alpha, ratio=1)

print(f"Achieved statistical power: {power:.4f}")
print(f"Effect size (Cohen's d): {abs(effect_size):.4f}")
print(f"Sample size: {n}")
print(f"Alpha level: {alpha}")

if power >= 0.8:
    print("✓ Sufficient power (≥ 0.8) to detect the observed effect")
else:
    print("⚠ Insufficient power (< 0.8) - may not reliably detect the effect")

mde = power_analysis.solve_power(power=0.8, nobs1=n, alpha=alpha, ratio=1)
print(f"Minimum detectable effect size (for 80% power): {mde:.4f}")

print("\n" + "="*60)
print("COMPREHENSIVE STATISTICAL SUMMARY")
print("="*60)

summary_stats = {
    "Test": ["McNemar's", "Paired T-Test", "Permutation", "Cohen's Kappa"],
    "Statistic": [
        f"{result.statistic:.4f} (χ²)", 
        f"{t_stat:.4f} (t)", 
        f"{obs_diff:.4f} (diff)", 
        f"{kappa:.4f}"
    ],
    "p-value": [
        f"{result.pvalue:.6f}", 
        f"{p_value:.6f}", 
        f"{perm_p_value:.6f}", 
        "N/A"
    ],
    "Significant": [
        "Yes" if result.pvalue < 0.05 else "No",
        "Yes" if p_value < 0.05 else "No",
        "Yes" if perm_p_value < 0.05 else "No",
        "N/A"
    ],
    "Effect Size": [
        f"OR={odds_ratio:.3f}",
        f"d={cohen_d:.3f}",
        f"Diff={obs_diff:.3f}",
        f"κ={kappa:.3f}"
    ]
}

summary_df = pd.DataFrame(summary_stats)
print("\nSummary of All Statistical Tests:")
print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("SAVING STATISTICAL ANALYSIS RESULTS")
print("="*60)

statistical_results = {
    'test_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_samples': n,
    'stage1_accuracy': float(stage1_accuracy),
    'stage2_accuracy': float(stage2_accuracy),
    'accuracy_difference': float(stage2_accuracy - stage1_accuracy),
    'mcnemar_chi2': float(result.statistic),
    'mcnemar_pvalue': float(result.pvalue),
    'mcnemar_significant': result.pvalue < 0.05,
    'paired_t_statistic': float(t_stat),
    'paired_t_pvalue': float(p_value),
    'paired_t_significant': p_value < 0.05,
    'mean_difference': float(mean_diff),
    'ci_lower': float(ci_lower),
    'ci_upper': float(ci_upper),
    'cohens_kappa': float(kappa),
    'cohens_d': float(cohen_d),
    'odds_ratio': float(odds_ratio),
    'permutation_pvalue': float(perm_p_value),
    'permutation_significant': perm_p_value < 0.05,
    'bootstrap_ci_lower': float(lower_ci),
    'bootstrap_ci_upper': float(upper_ci),
    'statistical_power': float(power),
    'minimum_detectable_effect': float(mde),
    'discordant_pairs': int(contingency_table[0,1] + contingency_table[1,0]),
    'stage2_corrections': int(contingency_table[0,1]),
    'stage2_regressions': int(contingency_table[1,0])
}

statistical_df = pd.DataFrame([statistical_results])
statistical_df.to_csv(
    "/home/aman_swaraj/Downloads/Codelite/codet5_statistical_analysis.csv",
    index=False
)

contingency_df = pd.DataFrame(
    contingency_table,
    index=['Stage1_Wrong', 'Stage1_Correct'],
    columns=['Stage2_Wrong', 'Stage2_Correct']
)
contingency_df.to_csv("/home/aman_swaraj/Downloads/Codelite/codet5_contingency_table.csv")

if significant_classes:
    per_class_df = pd.DataFrame(
        significant_classes,
        columns=['Class', 'Stage1_Accuracy', 'Stage2_Accuracy', 'Difference', 'p_value']
    )
    per_class_df.to_csv("/home/aman_swaraj/Downloads/Codelite/codet5_per_class_significance.csv", index=False)

print(f"✓ Statistical results saved to:")
print(f"  - codet5_statistical_analysis.csv")
print(f"  - codet5_contingency_table.csv")
if significant_classes:
    print(f"  - codet5_per_class_significance.csv")

print("\n" + "="*60)
print("KEY INTERPRETATION")
print("="*60)

print(f"\nBased on all statistical tests:")
print(f"1. Primary test (McNemar's): {'SIGNIFICANT' if result.pvalue < 0.05 else 'NOT SIGNIFICANT'}")
print(f"2. Effect direction: {'Stage 2 BETTER' if stage2_accuracy > stage1_accuracy else 'Stage 1 BETTER'}")
print(f"3. Effect size: {d_interpretation}")
print(f"4. Practical improvement: {stage2_accuracy - stage1_accuracy:.2%}")

if result.pvalue < 0.05 and stage2_accuracy > stage1_accuracy:
    print(f"\n✓ CONCLUSION: Your RI extraction approach provides statistically significant improvement!")
elif result.pvalue >= 0.05 and stage2_accuracy > stage1_accuracy:
    print(f"\n⚠ CONCLUSION: Improvement observed but not statistically significant at p<0.05")
else:
    print(f"\n✗ CONCLUSION: No statistically significant improvement detected")

print("\n" + "="*60)
print("STATISTICAL ANALYSIS COMPLETE")
print("="*60)